In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import PreTrainedTokenizerFast, AutoTokenizer

TOKENIZER_DIR  = "../model/wordlevel_tokenizer"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR, add_bos_token=True)

In [ ]:
import itertools
import json
from typing import Dict, List, Literal

def generate_task_prompts(dataset):
    task_prompts_next = {}
    flipped_prompts_next = {}
    task_prompts_last = {}
    flipped_prompts_last = {}

    # Step 1: Ascending/Descending sequences
    for task_name, values in dataset.items():
        for length in range(6, 11):
            if len(values) >= length:
                for i in range(len(values) - length + 1):
                    seq = values[i : i + length]
                    rev_seq = list(reversed(seq))

                    key_next = f"{task_name}/asc_{i}_{length}"
                    key_last = f"{task_name}/desc_{i}_{length}"

                    task_prompts_next[key_next] = " ".join(seq[:-1])
                    flipped_prompts_next[key_next] = " ".join(rev_seq[:-1])

                    task_prompts_last[key_last] = " ".join(seq)
                    flipped_prompts_last[key_last] = " ".join(rev_seq)

    # Step 2: Alternating sequences
    keys = list(dataset.keys())
    for key1, key2 in itertools.permutations(keys, 2):
        values1 = dataset[key1]
        values2 = dataset[key2]
        min_len = min(len(values1), len(values2))
        max_possible = min(min_len, 10)
        for length in range(6, max_possible + 1):
            for i in range(min_len - length + 1):
                seq1 = values1[i : i + length]
                seq2 = values2[i : i + length]
                alt_seq = list(itertools.chain.from_iterable(zip(seq1, seq2)))
                alt_seq_rev = list(reversed(alt_seq))

                task_id = f"{key1}_vs_{key2}/alt_{i}_{length}"

                task_prompts_next[task_id] = " ".join(alt_seq[:-1])
                flipped_prompts_next[task_id] = " ".join(alt_seq_rev[:-1])

                task_prompts_last[task_id] = " ".join(alt_seq)
                flipped_prompts_last[task_id] = " ".join(alt_seq_rev)

    return task_prompts_next, flipped_prompts_next, task_prompts_last, flipped_prompts_last

In [ ]:
from data.data_template import succession_dataset
from data.succession import create_augmented_prompts, to_json_format

task_prompts_next, flipped_prompts_next, task_prompts_last, flipped_prompts_last = generate_task_prompts(succession_dataset)

aug_next = create_augmented_prompts(
    tokenizer,
    task_prompts_next,
    flipped_prompts_next,
    truncate=False,
    which_task="next",
)

aug_last = create_augmented_prompts(
    tokenizer,
    task_prompts_last,
    flipped_prompts_last,
    truncate=False,
    which_task="last",
)

to_json_format(aug_next, save_path="succession_augmented_next_big.json")
to_json_format(aug_last, save_path="succession_augmented_last_big.json")

combined = aug_next + aug_last
to_json_format(combined, save_path="succession_augmented_both_big.json")